In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Project paths
PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
DATA_PATH = PROJECT_DIR / "data" / "processed" / "retail_sales_final.csv"
EDA_DIR = PROJECT_DIR / "reports" / "eda"

EDA_DIR.mkdir(parents=True, exist_ok=True)

# Load the current master dataset
df_eda = pd.read_csv(
    DATA_PATH,
    low_memory=False,
    dtype={"StockCode": "string"}
)

# Check required columns
required_columns = [
    "Invoice", "StockCode", "Description",
    "Quantity", "InvoiceDate", "Price",
    "Customer ID", "Country"
]

missing_columns = [
    col for col in required_columns
    if col not in df_eda.columns
]

if missing_columns:
    raise ValueError(f"Required columns missing: {missing_columns}")

# Parse dates for analysis only
raw_dates = df_eda["InvoiceDate"].copy()

df_eda["InvoiceDate"] = pd.to_datetime(
    raw_dates,
    format="%Y-%m-%d",
    errors="coerce"
)

# Convert numerical fields for analysis
for col in ["Quantity", "Price", "Revenue"]:
    if col in df_eda.columns:
        df_eda[col] = pd.to_numeric(
            df_eda[col], errors="coerce"
        )

# Use Revenue when available; otherwise calculate it for analysis
if "Revenue" in df_eda.columns:
    REVENUE_COL = "Revenue"
else:
    df_eda["_EDA_Revenue"] = (
        df_eda["Quantity"] * df_eda["Price"]
    )
    REVENUE_COL = "_EDA_Revenue"

print("=" * 60)
print("RETAILPULSE - EXPLORATORY DATA ANALYSIS")
print("=" * 60)

print("\nDataset shape:", df_eda.shape)
print("\nColumns:")
print(df_eda.columns.tolist())

print("\nFirst five rows:")
print(df_eda.head().to_string(index=False))

print("\nData types:")
print(df_eda.dtypes.to_string())

# Missing values
missing_summary = pd.DataFrame({
    "Missing_Count": df_eda.isna().sum(),
    "Missing_Percentage": (
        df_eda.isna().mean() * 100
    ).round(2)
}).sort_values("Missing_Count", ascending=False)

print("\nMissing values:")
print(missing_summary.to_string())

# Data quality checks
description_missing = (
    df_eda["Description"].isna().sum()
    + df_eda["Description"].fillna("").str.strip().eq("").sum()
)

stockcode_valid = df_eda["StockCode"].str.fullmatch(
    r"\d{5}", na=False
)

date_invalid = (
    raw_dates.notna()
    & raw_dates.astype("string").str.strip().ne("")
    & df_eda["InvoiceDate"].isna()
)

constant_columns = [
    col for col in df_eda.columns
    if df_eda[col].nunique(dropna=False) <= 1
]

quality_summary = {
    "Rows": len(df_eda),
    "Columns": len(df_eda.columns),
    "Exact_Duplicate_Rows": int(df_eda.duplicated().sum()),
    "Missing_Descriptions": int(description_missing),
    "Missing_StockCodes": int(df_eda["StockCode"].isna().sum()),
    "StockCodes_Exactly_5_Digits": int(stockcode_valid.sum()),
    "StockCodes_Not_5_Digits": int((~stockcode_valid).sum()),
    "Invalid_Dates": int(date_invalid.sum()),
    "Negative_Quantities": int((df_eda["Quantity"] < 0).sum()),
    "Zero_Quantities": int((df_eda["Quantity"] == 0).sum()),
    "Negative_Prices": int((df_eda["Price"] < 0).sum()),
    "Zero_Prices": int((df_eda["Price"] == 0).sum()),
    "Negative_Revenue_Rows": int(
        (df_eda[REVENUE_COL] < 0).sum()
    ),
    "Constant_Columns": ", ".join(constant_columns) or "None"
}

quality_df = pd.DataFrame(
    list(quality_summary.items()),
    columns=["Metric", "Value"]
)

print("\nDATA QUALITY SUMMARY")
print(quality_df.to_string(index=False))

# Save outputs for your project report
missing_summary.to_csv(
    EDA_DIR / "missing_values.csv"
)

quality_df.to_csv(
    EDA_DIR / "data_quality_summary.csv",
    index=False
)

df_eda.select_dtypes(include="number").describe().to_csv(
    EDA_DIR / "numeric_summary.csv"
)

print("\nEDA output folder:", EDA_DIR)


In [ ]:
import pandas as pd
import numpy as np

print("=" * 65)
print("RETAILPULSE - DATA QUALITY INVESTIGATION")
print("=" * 65)

# 1. Check duplicate transactions
duplicate_mask = df_eda.duplicated(keep=False)
duplicate_rows = df_eda.loc[duplicate_mask].copy()

print("\n1. DUPLICATE ROWS")
print("Exact duplicates beyond first occurrence:",
      int(df_eda.duplicated().sum()))
print("Rows involved in duplicate groups:",
      len(duplicate_rows))

duplicate_rows.head(5000).to_csv(
    EDA_DIR / "duplicate_rows_review.csv",
    index=False
)

# 2. Check return flags
negative_quantity = df_eda["Quantity"] < 0

print("\n2. RETURN CHECK")
print("Negative quantities:", int(negative_quantity.sum()))
print("Is_Return counts:")
print(df_eda["Is_Return"].value_counts(dropna=False))

return_comparison = pd.crosstab(
    negative_quantity.rename("Negative_Quantity"),
    df_eda["Is_Return"].rename("Is_Return")
)

print("\nNegative Quantity vs Is_Return:")
print(return_comparison)

return_flag = df_eda["Is_Return"].astype("boolean")
mismatch_mask = (
    return_flag.notna()
    & return_flag.ne(negative_quantity)
)

print("Return flag mismatches:", int(mismatch_mask.sum()))

df_eda.loc[mismatch_mask].head(1000).to_csv(
    EDA_DIR / "return_flag_mismatches.csv",
    index=False
)

# 3. Validate recorded revenue
expected_revenue = df_eda["Quantity"] * df_eda["Price"]

revenue_difference = (
    df_eda["Revenue"] - expected_revenue
).abs()

revenue_mismatch = (
    revenue_difference.gt(0.01)
    & df_eda["Revenue"].notna()
    & expected_revenue.notna()
)

print("\n3. REVENUE CHECK")
print("Rows where Revenue differs from Quantity * Price:",
      int(revenue_mismatch.sum()))
print("Negative recorded revenue:",
      int((df_eda["Revenue"] < 0).sum()))
print("Negative calculated revenue:",
      int((expected_revenue < 0).sum()))

df_eda.loc[revenue_mismatch].head(1000).to_csv(
    EDA_DIR / "revenue_mismatches.csv",
    index=False
)

# 4. Inspect zero-price transactions
zero_price_rows = df_eda[df_eda["Price"] == 0].copy()

print("\n4. ZERO-PRICE TRANSACTIONS")
print("Rows with zero price:", len(zero_price_rows))

print("\nMost common descriptions:")
print(
    zero_price_rows["Description"]
    .value_counts()
    .head(20)
    .to_string()
)

zero_price_rows.head(5000).to_csv(
    EDA_DIR / "zero_price_transactions.csv",
    index=False
)

# 5. Date coverage
valid_dates = df_eda["InvoiceDate"].dropna()

print("\n5. DATE COVERAGE")
print("Earliest:", valid_dates.min())
print("Latest:", valid_dates.max())
print("Unique dates:", valid_dates.nunique())

# 6. Save a summary for the project report
summary = pd.DataFrame([
    ("Dataset rows", len(df_eda)),
    ("Exact duplicates", int(df_eda.duplicated().sum())),
    ("Rows in duplicate groups", len(duplicate_rows)),
    ("Negative quantities", int(negative_quantity.sum())),
    ("Return flag mismatches", int(mismatch_mask.sum())),
    ("Zero-price rows", len(zero_price_rows)),
    ("Revenue mismatches", int(revenue_mismatch.sum())),
    ("Negative recorded revenue", int((df_eda["Revenue"] < 0).sum())),
    ("Negative calculated revenue", int((expected_revenue < 0).sum())),
    ("Unique dates", int(valid_dates.nunique()))
], columns=["Check", "Result"])

summary.to_csv(
    EDA_DIR / "data_quality_investigation.csv",
    index=False
)

print("\nINVESTIGATION SUMMARY")
print(summary.to_string(index=False))

print("\nReports saved to:", EDA_DIR)
print("Master CSV was not modified.")

In [ ]:
import pandas as pd

print("=" * 65)
print("RETAILPULSE - FINAL PRE-CHART DATA CHECKS")
print("=" * 65)

# 1. Check return rows and their prices
negative_qty = df_eda["Quantity"] < 0
zero_price = df_eda["Price"] == 0
positive_price = df_eda["Price"] > 0

print("\n1. NEGATIVE-QUANTITY ROWS")
print("Negative quantities:", int(negative_qty.sum()))
print("Negative quantity + zero price:",
      int((negative_qty & zero_price).sum()))
print("Negative quantity + positive price:",
      int((negative_qty & positive_price).sum()))

# 2. Check whether the zero-price flag is accurate
actual_zero_price = df_eda["Price"].eq(0)
stored_zero_flag = df_eda["Is_Zero_Price"].astype("boolean")

zero_flag_mismatch = (
    stored_zero_flag.isna()
    | stored_zero_flag.ne(actual_zero_price)
)

print("\n2. ZERO-PRICE FLAG")
print("Actual zero-price rows:", int(actual_zero_price.sum()))
print("Stored Is_Zero_Price counts:")
print(df_eda["Is_Zero_Price"].value_counts(dropna=False))
print("Flag mismatches:", int(zero_flag_mismatch.sum()))

# 3. Check whether the missing-description flag is stale
actual_missing_description = (
    df_eda["Description"].isna()
    | df_eda["Description"].astype("string").str.strip().eq("").fillna(False)
)

stored_missing_flag = df_eda["Missing_Description"].astype("boolean")

description_flag_mismatch = (
    stored_missing_flag.isna()
    | stored_missing_flag.ne(actual_missing_description)
)

print("\n3. MISSING-DESCRIPTION FLAG")
print("Actual missing or empty descriptions:",
      int(actual_missing_description.sum()))
print("Stored Missing_Description counts:")
print(df_eda["Missing_Description"].value_counts(dropna=False))
print("Flag mismatches:", int(description_flag_mismatch.sum()))

# 4. Save these findings for the report
checks = pd.DataFrame([
    ("Negative quantity rows", int(negative_qty.sum())),
    ("Negative quantity with zero price", int((negative_qty & zero_price).sum())),
    ("Negative quantity with positive price", int((negative_qty & positive_price).sum())),
    ("Zero-price flag mismatches", int(zero_flag_mismatch.sum())),
    ("Missing-description flag mismatches", int(description_flag_mismatch.sum()))
], columns=["Check", "Count"])

checks.to_csv(
    EDA_DIR / "return_and_flag_checks.csv",
    index=False
)

print("\nCheck results saved to:", EDA_DIR / "return_and_flag_checks.csv")
print("Master CSV was not modified.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# RETAILPULSE - EDA CELL 3: VISUAL ANALYSIS
# Uses only the main project columns.
# Does not modify retail_sales_final.csv.
# ============================================================

print("=" * 65)
print("RETAILPULSE - DISTRIBUTIONS AND SALES TRENDS")
print("=" * 65)

EDA_DIR.mkdir(parents=True, exist_ok=True)

# Main project columns only
analysis_cols = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
    "Revenue"
]

eda = df_eda[analysis_cols].copy()

eda["InvoiceDate"] = pd.to_datetime(
    eda["InvoiceDate"], errors="coerce"
)

for col in ["Quantity", "Price", "Revenue"]:
    eda[col] = pd.to_numeric(eda[col], errors="coerce")

# ------------------------------------------------------------
# 1. DISTRIBUTIONS OF QUANTITY, PRICE, AND REVENUE
# ------------------------------------------------------------

print("\n1. NUMERICAL DISTRIBUTIONS")

distribution_summary = []

for col in ["Quantity", "Price", "Revenue"]:
    values = eda[col].dropna()

    if values.empty:
        print(f"Skipping {col}: no usable numeric values.")
        continue

    distribution_summary.append({
        "Column": col,
        "Count": int(values.count()),
        "Mean": values.mean(),
        "Median": values.median(),
        "Minimum": values.min(),
        "Maximum": values.max(),
        "Standard_Deviation": values.std()
    })

    # Clip only for visualization to make the main shape readable
    lower = values.quantile(0.01)
    upper = values.quantile(0.99)
    plot_values = values[values.between(lower, upper)]

    plt.figure(figsize=(9, 5))
    plt.hist(plot_values, bins=50)
    plt.title(f"{col} Distribution (1st–99th Percentiles)")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(
        EDA_DIR / f"distribution_{col.lower()}.png",
        dpi=150
    )
    plt.show()
    plt.close()

distribution_df = pd.DataFrame(distribution_summary)
distribution_df.to_csv(
    EDA_DIR / "distribution_summary.csv",
    index=False
)

print(distribution_df.round(2).to_string(index=False))

# ------------------------------------------------------------
# 2. TOP 10 COUNTRIES BY RECORDED REVENUE
# ------------------------------------------------------------

print("\n2. TOP COUNTRIES BY RECORDED REVENUE")

country_revenue = (
    eda.groupby("Country", dropna=False)["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

country_revenue.to_csv(
    EDA_DIR / "country_revenue.csv",
    header=["Recorded_Revenue"]
)

top_countries = country_revenue.head(10).sort_values()

plt.figure(figsize=(10, 6))
top_countries.plot(kind="barh")
plt.title("Top 10 Countries by Recorded Revenue")
plt.xlabel("Recorded Revenue")
plt.ylabel("Country")
plt.tight_layout()
plt.savefig(
    EDA_DIR / "top_10_countries_by_revenue.png",
    dpi=150
)
plt.show()
plt.close()

print(country_revenue.head(10).round(2).to_string())

# ------------------------------------------------------------
# 3. MONTHLY REVENUE TREND
# ------------------------------------------------------------

print("\n3. MONTHLY REVENUE TREND")

dated = eda.dropna(subset=["InvoiceDate"]).copy()
dated = dated.set_index("InvoiceDate").sort_index()

monthly_revenue = dated["Revenue"].resample("MS").sum()
monthly_revenue.index.name = "Month"

monthly_revenue.to_csv(
    EDA_DIR / "monthly_revenue.csv",
    header=["Recorded_Revenue"]
)

plt.figure(figsize=(12, 5))
monthly_revenue.plot()
plt.title("Monthly Recorded Revenue")
plt.xlabel("Month")
plt.ylabel("Recorded Revenue")
plt.tight_layout()
plt.savefig(
    EDA_DIR / "monthly_revenue_trend.png",
    dpi=150
)
plt.show()
plt.close()

print(monthly_revenue.round(2).to_string())

# ------------------------------------------------------------
# 4. MONTHLY TRANSACTION-ROW TREND
# ------------------------------------------------------------

print("\n4. MONTHLY TRANSACTION-ROW TREND")

monthly_rows = dated["Invoice"].resample("MS").size()
monthly_rows.index.name = "Month"

monthly_rows.to_csv(
    EDA_DIR / "monthly_transaction_rows.csv",
    header=["Transaction_Rows"]
)

plt.figure(figsize=(12, 5))
monthly_rows.plot()
plt.title("Monthly Transaction Rows")
plt.xlabel("Month")
plt.ylabel("Number of Rows")
plt.tight_layout()
plt.savefig(
    EDA_DIR / "monthly_transaction_rows.png",
    dpi=150
)
plt.show()
plt.close()

# ------------------------------------------------------------
# 5. CORRELATION HEATMAP
# ------------------------------------------------------------

print("\n5. NUMERICAL CORRELATION")

correlation_cols = ["Quantity", "Price", "Revenue"]
corr_data = eda[correlation_cols].dropna()
correlation = corr_data.corr()

correlation.to_csv(
    EDA_DIR / "correlation_matrix.csv"
)

fig, ax = plt.subplots(figsize=(7, 6))

heatmap = ax.imshow(
    correlation.to_numpy(),
    vmin=-1,
    vmax=1,
    aspect="auto"
)

ax.set_xticks(
    range(len(correlation.columns)),
    labels=correlation.columns,
    rotation=45,
    ha="right"
)

ax.set_yticks(
    range(len(correlation.index)),
    labels=correlation.index
)

for row in range(len(correlation.index)):
    for col in range(len(correlation.columns)):
        value = correlation.iloc[row, col]
        if pd.notna(value):
            ax.text(
                col, row, f"{value:.2f}",
                ha="center",
                va="center"
            )

ax.set_title("Correlation Heatmap: Quantity, Price, Revenue")
fig.colorbar(heatmap, ax=ax, label="Correlation")
fig.tight_layout()
fig.savefig(
    EDA_DIR / "correlation_heatmap.png",
    dpi=150
)
plt.show()
plt.close()

print(correlation.round(3).to_string())

# ------------------------------------------------------------
# 6. FINISH
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("EDA VISUAL ANALYSIS COMPLETED")
print("=" * 65)
print("Charts and summaries saved to:")
print(EDA_DIR)
print("\nMaster CSV was not modified.")


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Report output folder
EDA_DIR = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail\reports\eda"
)
EDA_DIR.mkdir(parents=True, exist_ok=True)

# Use only the main project columns
project_columns = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country", "Revenue"
]

report_df = df_eda[project_columns].copy()

report_df["InvoiceDate"] = pd.to_datetime(
    report_df["InvoiceDate"], errors="coerce"
)

# Key metrics
rows = len(report_df)
columns = len(project_columns)
missing_customers = int(report_df["Customer ID"].isna().sum())
missing_descriptions = int(
    report_df["Description"].isna().sum()
    + report_df["Description"].fillna("").str.strip().eq("").sum()
)

duplicates = int(report_df.duplicated().sum())
negative_qty = report_df["Quantity"] < 0
zero_price = report_df["Price"] == 0

revenue_expected = report_df["Quantity"] * report_df["Price"]
revenue_mismatch = int(
    (
        (report_df["Revenue"] - revenue_expected).abs() > 0.01
    ).sum()
)

date_min = report_df["InvoiceDate"].min()
date_max = report_df["InvoiceDate"].max()
unique_dates = int(report_df["InvoiceDate"].nunique())

# Summaries for interpretation
country_revenue = (
    report_df.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

monthly_revenue = (
    report_df.dropna(subset=["InvoiceDate"])
    .set_index("InvoiceDate")["Revenue"]
    .resample("MS")
    .sum()
)

peak_month = monthly_revenue.idxmax().strftime("%Y-%m")
peak_month_revenue = monthly_revenue.max()

correlation = report_df[
    ["Quantity", "Price", "Revenue"]
].corr()

stats = report_df[
    ["Quantity", "Price", "Revenue"]
].describe()

# Save summary data for later reporting
stats.to_csv(EDA_DIR / "report_numeric_statistics.csv")
correlation.to_csv(EDA_DIR / "report_correlation_matrix.csv")
country_revenue.to_csv(EDA_DIR / "report_country_revenue.csv")
monthly_revenue.to_csv(EDA_DIR / "report_monthly_revenue.csv")

# Draft a report-ready Markdown file
report_text = f"""# RetailPulse — Exploratory Data Analysis

## Dataset overview

The current processed master dataset contains {rows:,} transaction rows
and {columns} project-analysis columns.

The available dates range from {date_min:%Y-%m-%d} to
{date_max:%Y-%m-%d}, covering {unique_dates:,} unique dates.

## Data-quality findings

- Missing customer IDs: {missing_customers:,}
  ({missing_customers / rows * 100:.2f}% of rows).
- Missing or blank descriptions: {missing_descriptions:,}.
- Exact duplicate rows beyond the first occurrence, using the
  selected project columns: {duplicates:,}.
- Negative-quantity rows: {int(negative_qty.sum()):,}.
- Zero-price rows: {int(zero_price.sum()):,}.
- Rows where recorded Revenue differs from Quantity × Price
  by more than 0.01: {revenue_mismatch:,}.

Duplicate rows require review before they are removed. Missing customer
IDs limit customer-level analysis. Zero-price and negative-quantity
records require careful interpretation before forecasting or revenue
metrics are finalized.

## Distribution analysis

Quantity, Price and Revenue have right-skewed distributions. Their
means exceed their medians, indicating that larger values influence
the averages. The visualized histograms use the 1st through 99th
percentiles for readability; the underlying data was not clipped.

## Country analysis

The country with the highest recorded revenue is
{country_revenue.index[0]}, with recorded revenue of
{country_revenue.iloc[0]:,.2f}.

Revenue is concentrated in this country compared with other countries
in the dataset.

## Monthly trends

The highest monthly recorded revenue occurs in {peak_month}, at
{peak_month_revenue:,.2f}.

The data ends on {date_max:%Y-%m-%d}; December 2011 is incomplete and
should not be compared directly with complete months.

## Correlation analysis

- Quantity–Revenue correlation: {correlation.loc["Quantity", "Revenue"]:.3f}
- Price–Revenue correlation: {correlation.loc["Price", "Revenue"]:.3f}
- Quantity–Price correlation: {correlation.loc["Quantity", "Price"]:.3f}

These are descriptive associations, not causal effects. Because
Revenue is calculated as Quantity × Price, correlations involving
Revenue are not independent evidence of a business driver.

## Limitations and preprocessing notes

- In the current dataset, negative-quantity records have zero price.
  These records therefore contribute zero to calculated revenue, so
  recorded revenue may not reflect the monetary impact of returns.
- StockCode suffixes were removed during preprocessing, and rows
  whose StockCodes did not meet the five-digit rule were removed.
  Distinct original product codes may therefore have been combined.
- These findings describe the current processed dataset, not the
  untouched original source data.
- Model acceptance criteria have not been evaluated by this EDA alone;
  forecasting, segmentation, churn and inventory results must be
  rerun and assessed separately.
"""

report_path = EDA_DIR / "eda_findings.md"
report_path.write_text(report_text, encoding="utf-8")

print("EDA findings file saved:")
print(report_path)
print("\nFirst part of the report:")
print(report_text[:2500])

